## Day 2 - Part 3: 모델 평가와 검증: 신뢰할 수 있는 모델 만들기

### 개요

Day 2의 앞선 파트들에서 우리는 모델의 구조를 설계하고(`Part 1`), 다양한 하이퍼파라미터를 조정하여(`Part 2`) 모델의 성능을 끌어올리는 방법을 배웠습니다. 

하지만 여기서 중요한 질문이 남습니다. "우리가 만든 모델이 정말로 '좋은' 모델이라고 어떻게 확신할 수 있을까요?"

모델이 훈련 데이터에서 높은 점수를 받는 것은 마치 학생이 연습 문제의 답을 통째로 외워 시험을 잘 보는 것과 같습니다. 

진짜 실력은 처음 보는 문제, 즉 `새로운 데이터`에 얼마나 잘 대응하는지로 드러납니다.

이번 파트에서는 모델의 '진짜 실력'을 측정하고 신뢰성을 확보하는 데 필수적인 `모델 평가(Evaluation)와 검증(Validation)` 기법들을 깊이 있게 탐구합니다. 

우리는 모델에게 공정한 시험을 치르게 하고, 그 성적표를 올바르게 해석하는 방법을 배울 것입니다.

`이번 파트의 학습 목표:`

* 모델 학습의 가장 중요한 원칙인 `훈련(Train), 검증(Validation), 테스트(Test) 세트`의 역할을 이해하고 데이터를 올바르게 분리할 수 있습니다. 

* 학습 곡선(Learning Curve)을 통해 모델의 `과적합(Overfitting)` 과 `과소적합(Underfitting)` 상태를 진단할 수 있습니다. 
* 분류 모델의 성능을 정밀하게 측정하는 `혼동 행렬(Confusion Matrix)`을 이해하고, `정확도(Accuracy), 정밀도(Precision), 재현율(Recall), F1-Score`를 계산하고 해석할 수 있습니다. 
* 더 안정적이고 신뢰도 높은 모델 성능을 측정하기 위한 `교차 검증(Cross-Validation)` 기법을 이해하고 적용할 수 있습니다. 
* 훈련 과정에서 과적합을 방지하고 최적의 모델을 저장하는 `조기 종료(Early Stopping)`와 `모델 체크포인트(Model Checkpointing)`를 구현할 수 있습니다. 

이번 파트에서도 `위스콘신 유방암 데이터셋`을 사용하여, 우리가 배운 평가 기법들을 통해 모델의 성능을 객관적으로 분석하고 개선해 나가겠습니다.


### 1. 평가의 황금률: 훈련, 검증, 테스트 세트 분리

모델을 공정하게 평가하기 위한 가장 기본적인 단계는 데이터를 세 가지 목적에 맞게 나누는 것입니다: `훈련, 검증, 테스트`.

마치 수능을 준비하는 수험생과 같습니다.

* `훈련 세트 (Training Set)`: 개념을 익히고 문제 풀이를 연습하는 '교과서'와 '문제집'입니다.  모델은 이 데이터를 보고 패턴을 학습하고 파라미터(가중치와 편향)를 업데이트합니다. 전체 데이터의 약 60-80%를 차지합니다.

* `검증 세트 (Validation Set)`: 실력을 중간 점검하고 약점을 파악하기 위해 푸는 '모의고사'입니다.  모델의 학습이 잘 되어가는지, 과적합은 없는지 모니터링하고, 학습률, 은닉층의 수와 같은 하이퍼파라미터를 조정하는 기준으로 사용됩니다. 
* `테스트 세트 (Test Set)`: 모든 학습과 모의고사가 끝난 후 단 한 번만 응시하는 '수능 시험'입니다.  모델의 최종 성능을 평가하기 위해 사용되며, 이 데이터는 모델 훈련 및 하이퍼파라미터 조정 과정에 절대 사용되어서는 안 됩니다. 

이 세 가지로 데이터를 나누는 것이 모델의 일반화 성능을 가장 정직하게 측정하는 방법입니다.

#### 코드 실습: 데이터셋 분리하기

`scikit-learn`의 `train_test_split` 함수를 사용하면 손쉽게 데이터를 나눌 수 있습니다. 보통 전체 데이터를 훈련+검증 세트와 테스트 세트로 먼저 나눈 뒤, 다시 훈련 세트를 훈련과 검증으로 나눕니다.

In [ ]:
import torch
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. 데이터 로드
X, y = load_breast_cancer(return_X_y=True)

# 2. 훈련+검증 데이터와 테스트 데이터로 분리 (80% vs 20%)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. 훈련 데이터와 검증 데이터로 분리 (원본의 60% vs 20%)
# test_size=0.25는 X_train_val (전체의 80%) 중에서 25%를 의미 -> 전체의 20%
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)

# 4. 데이터 스케일링
# 훈련 데이터 기준으로 스케일러를 학습(fit)시키고 모두에게 적용
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test) # test set은 준비해둔 후 마지막에 한 번만 확인


print(f"전체 데이터 수: {len(X)}")
print(f"훈련 데이터 수: {len(X_train)} ({len(X_train)/len(X):.0%})")
print(f"검증 데이터 수: {len(X_val)} ({len(X_val)/len(X):.0%})")
print(f"테스트 데이터 수: {len(X_test)} ({len(X_test)/len(X):.0%})")

전체 데이터 수: 569
훈련 데이터 수: 341 (60%)
검증 데이터 수: 114 (20%)
테스트 데이터 수: 114 (20%)




### 2\. 모델의 성적표: 과적합과 과소적합 진단하기

모델을 훈련시키면 훈련 손실(train loss)과 검증 손실(validation loss)의 변화를 관찰할 수 있습니다. 이 두 곡선의 관계를 나타내는 \*\*학습 곡선(Learning Curve)\*\*은 모델의 상태를 진단하는 핵심적인 성적표입니다.

  * `과소적합 (Underfitting)`: 모델이 너무 단순하거나 훈련이 부족하여 훈련 데이터의 패턴조차 제대로 학습하지 못한 상태입니다. 
      
      * `진단`: 훈련 손실과 검증 손실이 `모두 높고`, 더 이상 감소하지 않습니다. 마치 공부를 너무 안 해서 연습 문제도, 모의고사도 모두 못 푸는 것과 같습니다. 
  
  * `과적합 (Overfitting)`: 모델이 훈련 데이터에만 너무 특화되어, 데이터의 실제 패턴을 넘어 노이즈까지 암기해버린 상태입니다. 
      * `진단`: 훈련 손실은 `계속 낮아지는데`, 어느 시점부터 검증 손실은 `더 이상 낮아지지 않거나 오히려 상승`합니다. 연습 문제집은 달달 외워 100점을 받지만, 처음 보는 모의고사는 망치는 상황입니다. 
  * `적절한 상태 (Good Fit)`: 훈련 손실과 검증 손실이 `모두 낮게 수렴`하고, 두 값의 차이가 크지 않습니다.

#### 코드 실습: 학습 곡선을 통해 과적합 관찰하기

Day 2-Part 1에서 만들었던 `AdvancedClassifier` 모델을 사용하여 학습 곡선을 시각화하고 과적합 현상을 직접 확인해 보겠습니다.


In [2]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import plotly.graph_objects as go

# 데이터셋 및 데이터로더 정의 (앞선 코드와 동일)
class BreastCancerDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)
    def __len__(self): return len(self.features)
    def __getitem__(self, idx): return self.features[idx], self.labels[idx]

train_dataset = BreastCancerDataset(X_train, y_train)
val_dataset = BreastCancerDataset(X_val, y_val)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [3]:
# 모델 정의 (Day 2-Part 1의 모델)
class AdvancedClassifier(nn.Module):
    def __init__(self, num_features, num_classes, dropout_p=0.4):
        super(AdvancedClassifier, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(num_features, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(p=dropout_p),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(p=dropout_p),
            nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(p=dropout_p),
            nn.Linear(32, num_classes)
        )
    def forward(self, x): return self.net(x)

In [4]:
# 학습 및 평가 함수 정의
def train_and_get_history(model, train_loader, val_loader, epochs=100):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    history = {'train_loss': [], 'val_loss': []}

    for epoch in range(epochs):
        # 훈련
        model.train()
        train_loss = 0.0
        for features, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        history['train_loss'].append(train_loss / len(train_loader))

        # 검증
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for features, labels in val_loader:
                outputs = model(features)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
        history['val_loss'].append(val_loss / len(val_loader))

    return history

In [5]:
# 모델 생성 및 학습
model = AdvancedClassifier(num_features=X_train.shape[1], num_classes=2)
history = train_and_get_history(model, train_loader, val_loader, epochs=150)

In [ ]:
# 학습 곡선 시각화
fig = go.Figure()
epoch_list = list(range(1, 151))
fig.add_trace(go.Scatter(x=epoch_list, y=history['train_loss'], name='Train Loss'))
fig.add_trace(go.Scatter(x=epoch_list, y=history['val_loss'], name='Validation Loss'))

# 과적합 지점 찾기 (검증 손실이 최소인 지점)
overfitting_epoch = np.argmin(history['val_loss']) + 1 # 최솟값이 있는 곳의 인덱스값을 가져와 + 1
min_val_loss = min(history['val_loss'])

# 과적합 지점에 수직선 추가
fig.add_vline(x=overfitting_epoch, line_dash="dash", line_color="red", 
              annotation_text=f"과적합 시작점 (Epoch {overfitting_epoch})",
              annotation_position="top right")

fig.update_layout(title='학습 곡선 (Learning Curve)', xaxis_title='Epochs', yaxis_title='Loss')
fig.show()

이 그래프를 보면, 초기에는 훈련 손실과 검증 손실이 함께 감소하다가 특정 시점 이후 훈련 손실은 계속 내려가지만 검증 손실은 정체되거나 오히려 상승하는 것을 볼 수 있습니다. 

이 지점이 바로 과적합이 시작되는 구간입니다.

### 3. 분류 모델의 현미경: 혼동 행렬과 성능 지표

"정확도 99%"라는 말은 항상 좋은 모델을 의미할까요? 만약 100명의 환자 중 1명만 암 환자인 데이터에서, 모델이 모든 환자를 '정상'이라고 예측한다면 정확도는 99%가 됩니다. 

하지만 정작 가장 중요한 암 환자는 단 한 명도 찾아내지 못했습니다. 

이처럼 정확도(Accuracy)는 클래스 분포가 불균형할 때 모델의 성능을 제대로 나타내지 못할 수 있습니다.  

이럴 때 우리는 `혼동 행렬(Confusion Matrix)` 이라는 현미경으로 모델의 예측을 더 자세히 들여다봐야 합니다.

혼동 행렬은 모델이 각 클래스를 얼마나 잘 맞췄고, 어떤 클래스와 헷갈렸는지(confused)를 표로 보여줍니다. 이진 분류(Positive/Negative)의 경우 4가지 경우로 나뉩니다.

  * `True Positive (TP)`: 실제 'Positive'를 'Positive'로 올바르게 예측 (암 환자를 암으로 진단) 
  
  * `True Negative (TN)`: 실제 'Negative'를 'Negative'로 올바르게 예측 (정상인을 정상으로 진단) 
  * `False Positive (FP)`: 실제 'Negative'를 'Positive'로 잘못 예측 (정상인을 암으로 오진, Type I Error) 
  * `False Negative (FN)`: 실제 'Positive'를 'Negative'로 잘못 예측 (암 환자를 정상으로 오진, Type II Error) 

이 네 가지 값을 기반으로 더 정교한 성능 지표들을 계산할 수 있습니다.

  * `정확도 (Accuracy)`: 전체 예측 중 올바르게 예측한 비율. $= \frac{TP+TN}{TP+TN+FP+FN}$ 
  
  * `정밀도 (Precision)`: 모델이 'Positive'라고 예측한 것들 중, 실제 'Positive'인 것의 비율. $= \frac{TP}{TP+FP}$ 
      
      * `중요할 때`: FP의 비용이 클 때. 예를 들어, 스팸 메일 필터에서 일반 메일을 스팸으로 분류(FP)하면 중요한 내용을 놓칠 수 있으므로 정밀도가 중요합니다. 
  
  * `재현율 (Recall / Sensitivity)`: 실제 'Positive'들 중, 모델이 'Positive'로 예측해낸 것의 비율. $= \frac{TP}{TP+FN}$ 
      
      * `중요할 때`: FN의 비용이 클 때. 예를 들어, 암 진단 모델에서 암 환자를 정상으로 분류(FN)하면 치명적이므로 재현율이 매우 중요합니다. 
  
  * `F1-Score`: 정밀도와 재현율의 조화 평균. 두 지표가 모두 중요할 때 사용합니다. $= 2 \times \frac{Precision \times Recall}{Precision + Recall}$ 

#### 코드 실습: 혼동 행렬 시각화 및 성능 지표 계산

학습된 모델을 테스트 데이터로 평가하고, `scikit-learn.metrics`를 이용해 혼동 행렬과 각종 성능 지표를 확인해 봅시다.

In [7]:
from sklearn.metrics import confusion_matrix, classification_report
import plotly.express as px

# 테스트 데이터셋 및 로더 생성
test_dataset = BreastCancerDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=len(test_dataset))

# 모델 평가
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for features, labels in test_loader:
        outputs = model(features)
        _, predicted = torch.max(outputs.data, 1)
        all_preds.extend(predicted.numpy())
        all_labels.extend(labels.numpy())

In [8]:
# 혼동 행렬 계산
# scikit-learn의 y_true, y_pred 순서를 따름
cm = confusion_matrix(all_labels, all_preds)

# 클래스 이름 (0: 악성, 1: 양성)
class_names = ['Malignant(악성)', 'Benign(양성)']

# 혼동 행렬 시각화
fig = px.imshow(cm,
                labels=dict(x="Predicted Class", y="Actual Class", color="Count"),
                x=class_names,
                y=class_names,
                text_auto=True,
                color_continuous_scale='Blues')
fig.update_layout(title='Confusion Matrix')
fig.show()

In [9]:
# 분류 리포트 출력
# target_names를 통해 각 클래스에 대한 지표를 확인
report = classification_report(all_labels, all_preds, target_names=class_names)
print(report)

               precision    recall  f1-score   support

Malignant(악성)       0.93      0.93      0.93        42
   Benign(양성)       0.96      0.96      0.96        72

     accuracy                           0.95       114
    macro avg       0.94      0.94      0.94       114
 weighted avg       0.95      0.95      0.95       114



### 4. 더 똑똑하게 평가하기: 교차 검증 (Cross-Validation)

우리가 데이터를 훈련/검증/테스트 세트로 한 번만 나누면, 그 결과는 특정 분할에 따른 '운'이 작용할 수 있습니다.  

우연히 쉬운 데이터가 검증 세트로 가거나, 어려운 데이터가 몰려갈 수 있기 때문입니다.

`K-겹 교차 검증 (K-Fold Cross-Validation)`은 이 문제를 해결하기 위한 기법입니다.  

훈련 데이터를 K개의 '겹(fold)'으로 나눈 뒤, K번의 훈련과 검증을 반복합니다. 각 반복마다 하나의 겹을 검증 세트로 사용하고 나머지 K-1개의 겹을 훈련 세트로 사용합니다. 

최종적으로 K개의 검증 점수를 평균내어 모델의 성능을 평가합니다. 이를 통해 특정 데이터 분할에 대한 의존도를 줄이고 훨씬 더 안정적이고 일반화된 성능 추정치를 얻을 수 있습니다. 

#### 코드 실습: PyTorch 모델에 대한 교차 검증

`scikit-learn`의 `KFold`와 PyTorch를 결합하여 교차 검증을 구현해 보겠습니다.

In [10]:
from sklearn.model_selection import KFold

# 교차 검증을 위한 설정
k_folds = 5
kfold = KFold(n_splits=k_folds, shuffle=True, random_state=42)

# 전체 훈련+검증 데이터를 사용
X_train_val_tensor = torch.FloatTensor(scaler.fit_transform(X_train_val))
y_train_val_tensor = torch.LongTensor(y_train_val)

cv_scores = []

# K-겹 교차 검증 루프
for fold, (train_ids, val_ids) in enumerate(kfold.split(X_train_val_tensor)):
    print(f"--- FOLD {fold+1}/{k_folds} ---")

    # 데이터 분할
    train_subsampler = torch.utils.data.SubsetRandomSampler(train_ids)
    val_subsampler = torch.utils.data.SubsetRandomSampler(val_ids)

    train_fold_loader = DataLoader(
        dataset=BreastCancerDataset(X_train_val, y_train_val),
        batch_size=32,
        sampler=train_subsampler
    )
    val_fold_loader = DataLoader(
        dataset=BreastCancerDataset(X_train_val, y_train_val),
        batch_size=32,
        sampler=val_subsampler
    )

    # 모델, 손실함수, 옵티마이저 초기화 (매 폴드마다 새로 생성)
    model_cv = AdvancedClassifier(num_features=X_train.shape[1], num_classes=2)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model_cv.parameters(), lr=0.001)

    # 학습 (간략하게 20 에포크만 진행)
    for epoch in range(20):
        model_cv.train()
        for features, labels in train_fold_loader:
            # 스케일링된 데이터 사용
            scaled_features = torch.FloatTensor(scaler.transform(features.numpy()))
            optimizer.zero_grad()
            outputs = model_cv(scaled_features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

    # 각 폴드의 성능 평가 (정확도)
    model_cv.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for features, labels in val_fold_loader:
            scaled_features = torch.FloatTensor(scaler.transform(features.numpy()))
            outputs = model_cv(scaled_features)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100.0 * correct / total
    cv_scores.append(accuracy)
    print(f"Fold {fold+1} Accuracy: {accuracy:.2f}%")


# 최종 결과
print(f"K-Fold 교차 검증 (k={k_folds})")
print(f"Scores: {cv_scores}")
print(f"Average Accuracy: {np.mean(cv_scores):.2f}% (+/- {np.std(cv_scores):.2f}%)")

--- FOLD 1/5 ---
Fold 1 Accuracy: 100.00%
--- FOLD 2/5 ---
Fold 2 Accuracy: 96.70%
--- FOLD 3/5 ---
Fold 3 Accuracy: 98.90%
--- FOLD 4/5 ---
Fold 4 Accuracy: 98.90%
--- FOLD 5/5 ---
Fold 5 Accuracy: 94.51%
K-Fold 교차 검증 (k=5)
Scores: [100.0, 96.7032967032967, 98.9010989010989, 98.9010989010989, 94.50549450549451]
Average Accuracy: 97.80% (+/- 1.97%)


### 5. 과적합 방지 실전 테크닉: 조기 종료와 모델 체크포인트

학습 곡선에서 보았듯이, 너무 오래 훈련하면 과적합이 발생합니다. 

`조기 종료(Early Stopping)`와 `모델 체크포인트(Model Checkpointing)`는 이를 방지하는 매우 효과적인 실전 테크닉입니다.

  * `조기 종료 (Early Stopping)`: 검증 세트의 손실(또는 성능 지표)을 계속 모니터링하다가, 특정 횟수(`patience`) 동안 성능 개선이 없으면 훈련을 조기에 중단하는 기법입니다. 케이크가 타기 전에 오븐에서 꺼내는 것과 같습니다. 
  
  * `모델 체크포인트 (Model Checkpointing)`: 훈련 과정 중 검증 성능이 가장 좋았을 때의 모델 파라미터(가중치)를 파일로 저장하는 것입니다. 조기 종료와 함께 사용하면, 학습이 중단되었을 때 가장 성능이 좋았던 '최고의 모델'을 확보할 수 있습니다. 

#### 코드 실습: 조기 종료 및 모델 체크포인트 구현하기

훈련 루프 안에 조기 종료와 체크포인트 로직을 추가하여 가장 최적화된 모델을 찾아보겠습니다.

In [11]:
import os

def train_with_early_stopping(model, train_loader, val_loader, epochs=150, patience=10, model_path='best_model.pth'):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

    best_val_loss = float('inf')
    epochs_no_improve = 0

    for epoch in range(epochs):
        # 훈련
        model.train()
        train_loss = 0.0
        for features, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        history['train_loss'].append(train_loss / len(train_loader))

        # 검증
        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for features, labels in val_loader:
                outputs = model(features)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        avg_val_loss = val_loss / len(val_loader)
        val_accuracy = 100 * correct / total
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(val_accuracy)

        print(f"에포크 [{epoch+1}/{epochs}] 훈련 손실: {history['train_loss'][-1]:.4f}, 검증 손실: {avg_val_loss:.4f}, 검증 정확도: {val_accuracy:.2f}%")

        # 조기 종료 및 모델 체크포인트 로직
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0
            # 모델 체크포인트: 최고의 모델 저장
            torch.save(model.state_dict(), model_path)
            print(f"  -> 검증 손실이 개선되었습니다. 모델을 {model_path}에 저장합니다.")
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f"\n{patience} 에포크 동안 개선이 없어 조기 종료가 실행되었습니다.")
            break

    print("훈련이 완료되었습니다.")
    # 저장된 최고의 모델 가중치를 다시 로드
    model.load_state_dict(torch.load(model_path))
    print(f"{model_path}에서 최고 모델 가중치를 불러왔습니다.")
    return model, history


# 모델 생성 및 조기 종료를 사용한 학습
model_es = AdvancedClassifier(num_features=X_train.shape[1], num_classes=2)
best_model, history_es = train_with_early_stopping(model_es, train_loader, val_loader)

# 이제 'best_model'을 사용하여 최종 테스트를 진행하면 됩니다.

에포크 [1/150] 훈련 손실: 0.6093, 검증 손실: 0.5533, 검증 정확도: 82.46%
  -> 검증 손실이 개선되었습니다. 모델을 best_model.pth에 저장합니다.
에포크 [2/150] 훈련 손실: 0.4153, 검증 손실: 0.4035, 검증 정확도: 91.23%
  -> 검증 손실이 개선되었습니다. 모델을 best_model.pth에 저장합니다.
에포크 [3/150] 훈련 손실: 0.3499, 검증 손실: 0.2945, 검증 정확도: 92.98%
  -> 검증 손실이 개선되었습니다. 모델을 best_model.pth에 저장합니다.
에포크 [4/150] 훈련 손실: 0.2869, 검증 손실: 0.2484, 검증 정확도: 92.98%
  -> 검증 손실이 개선되었습니다. 모델을 best_model.pth에 저장합니다.
에포크 [5/150] 훈련 손실: 0.2509, 검증 손실: 0.2213, 검증 정확도: 93.86%
  -> 검증 손실이 개선되었습니다. 모델을 best_model.pth에 저장합니다.
에포크 [6/150] 훈련 손실: 0.2151, 검증 손실: 0.1951, 검증 정확도: 94.74%
  -> 검증 손실이 개선되었습니다. 모델을 best_model.pth에 저장합니다.
에포크 [7/150] 훈련 손실: 0.2015, 검증 손실: 0.1724, 검증 정확도: 94.74%
  -> 검증 손실이 개선되었습니다. 모델을 best_model.pth에 저장합니다.
에포크 [8/150] 훈련 손실: 0.2127, 검증 손실: 0.1442, 검증 정확도: 96.49%
  -> 검증 손실이 개선되었습니다. 모델을 best_model.pth에 저장합니다.
에포크 [9/150] 훈련 손실: 0.1614, 검증 손실: 0.1379, 검증 정확도: 96.49%
  -> 검증 손실이 개선되었습니다. 모델을 best_model.pth에 저장합니다.
에포크 [10/150] 훈련 손실: 0.1555, 검증 손실: 0.1213, 검증 정확도: 97.3